In [4]:
############################################################################################
# Imports
############################################################################################

import cobra
#import memote
import json
import pandas as pd
import re
import os
from cobra.io import read_sbml_model
from cobra import Reaction, Metabolite

In [5]:
############################################################################################
# Paths
############################################################################################

working_dir = '/home/emma/Dokumente/thesis/Model_generation_curation/Curated_models'
model_dir = os.path.join(working_dir, "mb1_mdr_rdr_dp_lib")
report_dir = os.path.join(model_dir, 'MEMOTE_reports')

In [3]:
############################################################################################
# Configurations
############################################################################################

# Making sure the solver is set to cplex
cobra.Configuration().solver = "cplex"

In [4]:
############################################################################################
# Functions
############################################################################################

def create_reports(model_dir, report_dir):

    # Iterate through each model in the directory
    for model_file in os.listdir(model_dir):

        if model_file.endswith('.xml'):

            if f'{model_file[:-4]}_report.html' not in os.listdir(report_dir):

                # Read the model using cobrapy
                model = cobra.io.read_sbml_model(os.path.join(model_dir, model_file))

                # Test the model with MEMOTE
                model_results = memote.test_model(model, results=True)

                # Generate a html report
                model_report = memote.snapshot_report(model_results[1], html=True)

                # Save the report
                save_path = os.path.join(report_dir, f'{model_file[:-4]}_report.html')
                with open(save_path, 'w') as report_file:
                    report_file.write(model_report)

def extract_window_data(html_file):
    """Extract window.data dictionary from MEMOTE HTML report"""
    with open(html_file, 'r') as f:
        html_content = f.read()
	
    
    # Find window.data = {...}
    match = re.search(r'window\.data\s*=\s*({.*?});', html_content, re.DOTALL)
    if match:
        json_str = match.group(1)
        data_dict = json.loads(json_str)
        return data_dict
    else:
        raise ValueError("Could not find window.data in HTML file")

def summarize_reports(report_dir):
    data = []
    blocked_reactions_dict = {}
    for file in os.listdir(report_dir):
        if file.endswith('.html'):
            html_file = os.path.join(report_dir, file)
            data_dict = extract_window_data(html_file)
            blocked_reactions = data_dict["tests"]["test_blocked_reactions"]["data"]
            for reac in blocked_reactions:
                if reac not in blocked_reactions_dict.keys():
                    blocked_reactions_dict[reac] = [file]
                else:
                    blocked_reactions_dict[reac].append(file)
            #total_score = data_dict['score']['total_score']["data"]
            #data.append({'Model': file[:-12], 'Total Score': total_score})
    summary = pd.DataFrame(data)
    #summary.to_csv(os.path.join(report_dir, 'Summary_of_all_reports.csv'))
    return blocked_reactions_dict, data_dict

In [8]:
def check_growth(model):
    biomass_test = []
    blocked_reactions = {}  # dict to collect blocked reactions per model

    for met in model.reactions.Growth.reactants:
        # temporary demand reaction for this specific metabolite: "metabolite -> " (draining it from the system)
        with model as temporary_model:
            try:
                demand_rxn = temporary_model.add_boundary(met, type="demand")
                temporary_model.objective = demand_rxn
                solution = temporary_model.optimize()
                status = "Pass" if solution.objective_value > 1e-5 else "BLOCKED"
                biomass_test.append(
                    {
                        "Metabolite_ID": met.id,
                        "Metabolite_Name": met.name,
                        "Production_Flux": solution.objective_value,
                        "Status": status,
                    }
                )
            except Exception as e:
                status = f"Error: {str(e)}"
                biomass_test.append(
                    {
                        "Metabolite_ID": met.id,
                        "Metabolite_Name": met.name,
                        "Production_Flux": 0.0,
                        "Status": status,
                    }
                )

    df_test = pd.DataFrame(biomass_test)
    blocked_components = df_test[df_test["Status"] == "BLOCKED"]

    # collect blocked reactions for this model
    if not blocked_components.empty:
        blocked_reactions[model.id] = blocked_components["Metabolite_ID"].tolist()
    else:
        blocked_reactions[model.id] = []

    print(f"Found {len(blocked_components)} biomass components that cannot be synthesized for model {model.id}.")
    return blocked_components, blocked_reactions

In [9]:
def compare_pre_postcur(prec_m, postc_m, met_id): #check fluxes for metabolites of interest pre and post curation steps 
    post_r = []
    pre_r = []
    pre_rxn = prec_m.metabolites.get_by_id(met_id).reactions
    post_rxn = postc_m.metabolites.get_by_id(met_id).reactions

    for rxn in pre_rxn:
        pre_r.append(rxn.id)
    for rxn in post_rxn:
        post_r.append(rxn.id)

    shared_rxn_ids = list(set(pre_r) & set(post_r))
    added_rxns = list(set(post_r) - set(pre_r))
    removed_rxns = list(set(pre_r) - set(post_r))

    print(f"Pre: {len(pre_rxn)}, Post: {len(post_rxn)}, Shared: {len(shared_rxn_ids)}")
    if added_rxns:
        print(f"Added in curation:    {added_rxns}")
    if removed_rxns:
        print(f"Removed in curation:  {removed_rxns}")

    # Check if bounds of shared reactions changed
    pre_bounds = {rxn.id: rxn.bounds for rxn in pre_rxn if rxn.id in shared_rxn_ids}
    post_bounds = {rxn.id: rxn.bounds for rxn in post_rxn if rxn.id in shared_rxn_ids}

    changed_bounds = {
        rxn_id: (pre_bounds[rxn_id], post_bounds[rxn_id])
        for rxn_id in shared_rxn_ids
        if pre_bounds[rxn_id] != post_bounds[rxn_id]
    }
    if changed_bounds:
        print("Reactions with changed bounds:")
        for rxn_id, (pre_b, post_b) in changed_bounds.items():
            print(f"  {rxn_id}: {pre_b} -> {post_b}")

    return pre_r, post_r

In [10]:
def demand_reaction(model, list_problem_met_ids): #create demand reaction for metabolite to check whether it can ever be produced
    demand_flux_d = {}
    for met_id in list_problem_met_ids:
        with model as temp_model:
            try:
                met_obj = temp_model.metabolites.get_by_id(met_id)
                
                # isolated demand reaction
                demand_rxn = temp_model.add_boundary(met_obj, type="demand") #"A demand reaction is an irreversible reaction that consumes an intracellular metabolite"
                temp_model.objective = demand_rxn
                #print(model.demands)
                sol = temp_model.optimize()
                
                #print(f"Max production flux of {met_id:10}: {sol.objective_value:.4f}")
                demand_flux_d[met_id] = sol.objective_value
            except KeyError:
                print(f"Error: Metabolite '{met_id}' not found in the model.")
            except Exception as e:
                print(f"Error optimizing {met_id}: {e}")
    return demand_flux_d

In [11]:
def get_producing_reactions(model, list_problem_met_ids): #get producing reacitons of metabolites
    prec_dict = {}
    for met_id in list_problem_met_ids:
        prec_cache = []

        try:
            met_obj = model.metabolites.get_by_id(met_id)
            print(f"=================== Analyzing: {met_id} ===================")
            
            producing_count = 0
            
            # Loop through all reactions the metabolite participates in
            for rxn in met_obj.reactions:
                # Get the stoichiometric coefficient of metabolite in this reaction
                coefficient = rxn.get_coefficient(met_obj)
                
                can_produce_forward = coefficient > 0 and rxn.upper_bound > 0
                can_produce_backward = coefficient < 0 and rxn.lower_bound < 0
                
                if can_produce_forward or can_produce_backward:
                    producing_count += 1
                    if can_produce_forward:
                        precursors = [met.id for met in rxn.reactants]
                        prec_cache.append(precursors)
                    else:
                        precursors = [met.id for met in rxn.products]
                        prec_cache.append(precursors)

                    print(f"Reaction: {rxn.id}")
                    #print(f"  Formula: {rxn.reaction}")
                    #print(f"  Bounds:  ({rxn.lower_bound}, {rxn.upper_bound})")
                    print(f"Precursors needed: {', '.join(precursors)}")
                    #print("-" * 40)
            if producing_count == 0:
                print(f"Direct Gap Found: No active reactions are configured to produce {met_id}!")
                
        except KeyError:
            print(f"Error: Metabolite '{met_id}' not found in the model.")
        prec_dict[met_id] = prec_cache

        met_list = [met for met_l in prec_dict.values() for met_i in met_l for met in met_i]
    return prec_dict, met_list

In [12]:
def get_non_growing_mets(demand2flux_dict):
    non_growing_mets = []
    for met_id,flux in demand2flux_dict.items():
        if flux == 0.000:
            non_growing_mets.append(met_id)
    return non_growing_mets

In [13]:
def artificial_cytoplasm_addition(model, met_id): #artificially add metabolites to cytosol 
    if met_id.endswith("_c"):
        clean_met_id = met_id[:-2]
    elif met_id.endswith("_p"):
        clean_met_id = met_id[:-2]
    else:
        clean_met_id = met_id

    cyto_met_id = clean_met_id + "_c"
    reac_id_str = met_id + "_syn"
    
    reac = Reaction(reac_id_str)
    reac.name = f"Artificial synthesis of {clean_met_id}"
    reac.subsystem = "Synthetic"
    reac.lower_bound = 0.0  
    reac.upper_bound = 1000.0
    if cyto_met_id in model.metabolites:
        met_to_add = model.metabolites.get_by_id(cyto_met_id)
    else:
        met_to_add = Metabolite(
            id=cyto_met_id,
            name=f"{clean_met_id} (cytoplasm)",
            compartment="c"
        )
        model.add_metabolites([met_to_add])
        
    reac.add_metabolites({met_to_add: 5.0})
    return reac

In [14]:
def test_biomass_prec(model): #test all biomass precursors of a model with a demand reaction to see which metabolites can not be produced
    store_dict = {}
    demand_dict = {}
    biomass_mets = [met.id for met in model.reactions.Growth.reactants]
    for met_id in biomass_mets:
        with model as model:
            # create and add synthetic reaction
            test_r = artificial_cytoplasm_addition(model, met_id)
            model.add_reactions([test_r])
            
            print(f"Added Reaction: {model.reactions.get_by_id(test_r.id).build_reaction_string()}")
            
            # optimize model 
            sol = model.optimize()
            print(f"Optimization Status: {sol.status}")
            print(f"Objective Value: {sol.objective_value}") 
            
            current_demand = demand_reaction(model, biomass_mets)
            demand_dict[met_id] = current_demand  
            ng_mets = get_non_growing_mets(current_demand)
            store_dict[met_id] = [ng_mets, sol.status, sol.objective_value]
    return store_dict, demand_dict

In [15]:
# function to fix reversibility of reactions if this lead to major changes in the objective value

def check_influence_reversibility(model, make_irreversible_rxns):
    reactions2change = []
    with model:
            #check reversibility of reactions by comparing objective value pre and post reversibility 
            for rxn_id in make_irreversible_rxns:
                #print(rxn_id)
                sol1 = model.optimize()
                if rxn_id in model.reactions:
                    model.reactions.get_by_id(rxn_id).bounds = (-1000, 1000)
                    sol2 = model.optimize()
                    diff_objv = abs(sol2.objective_value - sol1.objective_value) #absolute difference 
                    threshold = 0.1 * sol1.objective_value #if sol2 differs from sol1 by more than 10%
                    if diff_objv > threshold: #TODO: add two flags: one for more than 10% change, the other for 0 growth!!!
                        print(f"{model.id}: changing {rxn_id}, changes objective value by: {diff_objv}")
                        reactions2change.append(rxn_id)
    for rxn_id in reactions2change:
         model.reactions.get_by_id(rxn_id).bounds = (-1000, 1000) #if changing the reversibility alters the objective value by more than the threshold, the reactions stays reversible
                    

## check biomass synthesis

Problems BM syn:
- irreversible reactions 


In [ ]:
all_blocked = {}
make_rev = ["AADb", "APAT_1", "NNATr", "UDPACGLP", "NAPRT", "OXACOAL"]
for file in os.listdir(model_dir):
    if file.endswith('_lib.xml'):
        model = read_sbml_model(os.path.join(model_dir,file))
        with model:
            #check reversibility of reactions
            for rxn_id in make_rev:
                #print(rxn_id)
                sol1 = model.optimize()
                if rxn_id in model.reactions:
                    model.reactions.get_by_id(rxn_id).bounds = (-1000, 1000)
                    sol2 = model.optimize()
                    diff_objv = sol2.objective_value - sol1.objective_value
                    if diff_objv != 0:
                        print(f"{model.id}: changing {rxn_id}, changes objective value: {diff_objv}")

             # add free ADP source
            adp_source = model.add_boundary(model.metabolites.adp_c, type="sink")
            adp_source.lower_bound = -0.001  # add free ADP input
            
            # ATP demand reaction 
            atp_demand = model.add_boundary(model.metabolites.atp_c, type="demand")
            model.objective = atp_demand
            
            sol = model.optimize()
            print(f"ATP production flux with free ADP: {sol.objective_value}")
            
            blocked_components, blocked_reactions = check_growth(model)
            all_blocked.update(blocked_reactions)


ATP production flux with free ADP: 0.5594958101095955
Found 0 biomass components that cannot be synthesized for model m_1334.
m_2751: changing OXACOAL, changes objective value: 16.772360503419875
ATP production flux with free ADP: 0.7866457784115601
Found 0 biomass components that cannot be synthesized for model m_2751.
ATP production flux with free ADP: 0.5150866569390917
Found 0 biomass components that cannot be synthesized for model m_761.
ATP production flux with free ADP: 5.520720276691373
Found 0 biomass components that cannot be synthesized for model m_1252.
ATP production flux with free ADP: 0.3171874193918103
Found 0 biomass components that cannot be synthesized for model m_163.
ATP production flux with free ADP: 0.00025
Found 12 biomass components that cannot be synthesized for model m_428.
ATP production flux with free ADP: 0.0
Found 21 biomass components that cannot be synthesized for model m_1338.
ATP production flux with free ADP: 0.0
Found 22 biomass components that cann

In [ ]:
store_blocked = []
for mod_id, met_id in all_blocked.items():
    if len(met_id) > 0:
        for metabolite in met_id:
            store_blocked.append({
                "Model_ID": mod_id,
                "Blocked_Metabolite": metabolite
            })
store_blocked_df = pd.DataFrame(store_blocked)

In [ ]:
from collections import defaultdict

metabolites_blocked_in = defaultdict(list)

for mod_id, met_list in all_blocked.items():
    for metabolite in met_list:
        metabolites_blocked_in[metabolite].append(mod_id)

store_blocked = pd.DataFrame.from_dict(metabolites_blocked_in, orient='index')

store_blocked.columns = [f"Model_{i+1}" for i in range(store_blocked.shape[1])]

In [ ]:
filepath = os.path.join(working_dir, 'blocked_biomass_reactions_1006_1517.csv')

store_blocked.to_csv(filepath)

##### mql8 stuff

In [ ]:
mql_prec = ["2dmmq8_c", "amet_c"]
demand_reaction(m2751, mql_prec)

Max production flux of 2dmmq8_c  : 0.0000
Max production flux of amet_c    : 0.0000


In [ ]:
mql_prec2 = ["dhna_c", "octdp_c", "met__L_c"]
demand_reaction(m2751, mql_prec2)

Max production flux of dhna_c    : 500.0000
Max production flux of octdp_c   : 0.0000
Max production flux of met__L_c  : 0.0000


In [ ]:
mql_prec3 = ["frdp_c", "ipdp_c", "hcys__L_c", "mmet_c"]
demand_reaction(m2751, mql_prec3)

Max production flux of frdp_c    : 0.0000
Max production flux of ipdp_c    : 0.0000
Max production flux of hcys__L_c : 0.0000
Max production flux of mmet_c    : 0.0000


{'frdp_c': 0.0, 'ipdp_c': 0.0, 'hcys__L_c': 0.0, 'mmet_c': 0.0}

In [ ]:
mql_prec4d, mql_prec4l = get_producing_reactions(m2751, mql_prec3)
demand_mqlprec4l = demand_reaction(m2751, mql_prec4l)

=================== Analyzing: frdp_c ===================
Reaction: GRTT
Precursors needed: grdp_c, ipdp_c
=================== Analyzing: ipdp_c ===================
Reaction: DPMVD
Precursors needed: 5dpmev_c, atp_c
Reaction: IPDDI
Precursors needed: dmpp_c
=================== Analyzing: hcys__L_c ===================
Reaction: HCYStabcpp
Precursors needed: atp_c, h2o_c, hcys__L_p
Reaction: AHCi
Precursors needed: ahcys_c, h2o_c
=================== Analyzing: mmet_c ===================
Reaction: MMETt2pp
Precursors needed: h_p, mmet_p
Max production flux of grdp_c    : 0.0000
Max production flux of ipdp_c    : 0.0000
Max production flux of 5dpmev_c  : 0.0000
Max production flux of atp_c     : 0.0000
Max production flux of dmpp_c    : 0.0000
Max production flux of atp_c     : 0.0000
Max production flux of h2o_c     : 1000.0000
Max production flux of hcys__L_p : 1000.0000
Max production flux of ahcys_c   : 0.0000
Max production flux of h2o_c     : 1000.0000
Max production flux of h_p     

In [ ]:
mql_prec4l_0f = [met_id for met_id, flux in demand_mqlprec4l.items() if flux == 0.0] #only get fluxes that are 0 for further downstream ana
mql_prec5d, mql_prec5l = get_producing_reactions(m2751, mql_prec4l_0f)
demand_mqlprec5l = demand_reaction(m2751, mql_prec5l)


=================== Analyzing: grdp_c ===================
Reaction: DMATT
  Precursors needed: dmpp_c, ipdp_c

=================== Analyzing: ipdp_c ===================
Reaction: DPMVD
  Precursors needed: 5dpmev_c, atp_c
Reaction: IPDDI
  Precursors needed: dmpp_c

=================== Analyzing: 5dpmev_c ===================
Reaction: PMEVK
  Precursors needed: 5pmev_c, atp_c

=================== Analyzing: atp_c ===================
Reaction: GK1
  Precursors needed: adp_c, gdp_c
Reaction: NDPK4
  Precursors needed: adp_c, dttp_c
Reaction: NDPK2
  Precursors needed: adp_c, utp_c
Reaction: ATPS4rpp
  Precursors needed: adp_c, h_p, pi_c
Reaction: PRPPS
  Precursors needed: amp_c, h_c, prpp_c
Reaction: DTMPK
  Precursors needed: adp_c, dtdp_c
Reaction: ALAALAr
  Precursors needed: adp_c, alaala_c, h_c, pi_c
Reaction: PPK50r
  Precursors needed: adp_c, ppi50_c
Reaction: PGK
  Precursors needed: 13dpg_c, adp_c
Reaction: UMPK
  Precursors needed: adp_c, udp_c

=================== Analyzing:

In [ ]:
mql_prec5l_0f = [met_id for met_id, flux in demand_mqlprec5l.items() if flux == 0.0]
mql_prec6d, mql_prec6l = get_producing_reactions(m2751, mql_prec5l_0f)
demand_mqlprec6l = demand_reaction(m2751, mql_prec6l)

=================== Analyzing: dmpp_c ===================
Reaction: IPDDI
Precursors needed: ipdp_c
=================== Analyzing: ipdp_c ===================
Reaction: DPMVD
Precursors needed: 5dpmev_c, atp_c
Reaction: IPDDI
Precursors needed: dmpp_c
=================== Analyzing: 5dpmev_c ===================
Reaction: PMEVK
Precursors needed: 5pmev_c, atp_c
=================== Analyzing: atp_c ===================
Reaction: GK1
Precursors needed: adp_c, gdp_c
Reaction: NDPK4
Precursors needed: adp_c, dttp_c
Reaction: NDPK2
Precursors needed: adp_c, utp_c
Reaction: ATPS4rpp
Precursors needed: adp_c, h_p, pi_c
Reaction: PRPPS
Precursors needed: amp_c, h_c, prpp_c
Reaction: DTMPK
Precursors needed: adp_c, dtdp_c
Reaction: ALAALAr
Precursors needed: adp_c, alaala_c, h_c, pi_c
Reaction: PPK50r
Precursors needed: adp_c, ppi50_c
Reaction: PGK
Precursors needed: 13dpg_c, adp_c
Reaction: UMPK
Precursors needed: adp_c, udp_c
=================== Analyzing: 5pmev_c ===================
Reaction: ME

In [ ]:
mql_prec6l_0f = [met_id for met_id, flux in demand_mqlprec6l.items() if flux == 0.0]
mql_prec7d, mql_prec7l = get_producing_reactions(m2751, mql_prec6l_0f)
demand_mqlprec6l = demand_reaction(m2751, mql_prec7l)

## go through models step by step

1. in which curation step does the problem occur?
2. get all biomass precursors 
3. add them from nothing to cytosol
    - which precursors are still missing?
    - where in the curation process are they edited?

### load all models

##### m_2751

In [31]:
pcm2751 = read_sbml_model("/home/emma/Dokumente/thesis/Model_generation_curation/Draft_models/Models/2751.xml")
pmb2751 = read_sbml_model("/home/emma/Dokumente/thesis/Model_generation_curation/Curated_models/mb1/2751_or_mb1.xml") #post mass balance
pmc2751 = read_sbml_model("/home/emma/Dokumente/thesis/Model_generation_curation/Curated_models/mb1_mdr_rdr_dp_lib/2751_or_mb1_mdr_rdr_dp_lib.xml") #post macaw

Adding exchange reaction EX_LalaDgluMdapDala_e with default bounds for boundary metabolite: LalaDgluMdapDala_e.
Adding exchange reaction EX_acald_e with default bounds for boundary metabolite: acald_e.
Adding exchange reaction EX_acgam1p_e with default bounds for boundary metabolite: acgam1p_e.
Adding exchange reaction EX_acmana_e with default bounds for boundary metabolite: acmana_e.
Adding exchange reaction EX_akg_e with default bounds for boundary metabolite: akg_e.
Adding exchange reaction EX_alaala_e with default bounds for boundary metabolite: alaala_e.
Adding exchange reaction EX_alagly_e with default bounds for boundary metabolite: alagly_e.
Adding exchange reaction EX_amp_e with default bounds for boundary metabolite: amp_e.
Adding exchange reaction EX_arg__L_e with default bounds for boundary metabolite: arg__L_e.
Adding exchange reaction EX_asn__L_e with default bounds for boundary metabolite: asn__L_e.
Adding exchange reaction EX_asp__D_e with default bounds for boundary me

In [32]:
biomass_prec_2751 = [met.id for met in pcm2751.reactions.Growth.reactants]

In [34]:
for met_id in biomass_prec_2751:
    compare_pre_postcur(pcm2751, pmc2751, met_id)

Pre: 2, Post: 2, Shared: 2
Pre: 4, Post: 4, Shared: 4
Pre: 3, Post: 3, Shared: 3
Pre: 3, Post: 3, Shared: 3
Pre: 3, Post: 3, Shared: 3
Pre: 5, Post: 5, Shared: 5
Pre: 44, Post: 44, Shared: 43
Added in curation:    ['RBFK']
Removed in curation:  ['RBFK_1']
Reactions with changed bounds:
  NNATr: (-1000.0, 1000.0) -> (0.0, 1000.0)
  OXACOAL: (-1000.0, 1000.0) -> (0.0, 1000.0)
Pre: 2, Post: 2, Shared: 2
Pre: 2, Post: 2, Shared: 2
Pre: 12, Post: 12, Shared: 12
Reactions with changed bounds:
  OXACOAL: (-1000.0, 1000.0) -> (0.0, 1000.0)
Pre: 2, Post: 2, Shared: 2
Pre: 3, Post: 3, Shared: 3
Pre: 2, Post: 2, Shared: 2
Pre: 2, Post: 2, Shared: 2
Pre: 2, Post: 2, Shared: 2
Pre: 2, Post: 2, Shared: 2
Pre: 2, Post: 2, Shared: 2
Pre: 3, Post: 3, Shared: 3
Pre: 5, Post: 5, Shared: 5
Pre: 3, Post: 3, Shared: 3
Pre: 4, Post: 4, Shared: 4
Pre: 8, Post: 8, Shared: 8
Pre: 7, Post: 7, Shared: 7
Pre: 3, Post: 3, Shared: 3
Pre: 3, Post: 3, Shared: 3
Pre: 46, Post: 46, Shared: 46
Pre: 2, Post: 2, Shared: 2


In [ ]:
make_rev = ["OXACOAL", "NAPRT", "NNATr"] #check if chaningg reversibility lets it grow :) 
with m2751 as model:
    for rxn_id in make_rev:
        sol = model.optimize()
        print(sol.objective_value)
        model.reactions.get_by_id(rxn_id).bounds = (-1000, 1000)
        sol = model.optimize()
        print(sol.objective_value)

##### m_428

In [35]:
pcm428 = read_sbml_model("/home/emma/Dokumente/thesis/Model_generation_curation/Draft_models/Models/428.xml") #pre curation 

Adding exchange reaction EX_6apa_e with default bounds for boundary metabolite: 6apa_e.
Adding exchange reaction EX_6pgc_e with default bounds for boundary metabolite: 6pgc_e.
Adding exchange reaction EX_LalaDgluMdap_e with default bounds for boundary metabolite: LalaDgluMdap_e.
Adding exchange reaction EX_R_3httdca_e with default bounds for boundary metabolite: R_3httdca_e.
Adding exchange reaction EX_ac_e with default bounds for boundary metabolite: ac_e.
Adding exchange reaction EX_acald_e with default bounds for boundary metabolite: acald_e.
Adding exchange reaction EX_acmana_e with default bounds for boundary metabolite: acmana_e.
Adding exchange reaction EX_acnam_e with default bounds for boundary metabolite: acnam_e.
Adding exchange reaction EX_ala__L_e with default bounds for boundary metabolite: ala__L_e.
Adding exchange reaction EX_alaala_e with default bounds for boundary metabolite: alaala_e.
Adding exchange reaction EX_amp_e with default bounds for boundary metabolite: amp

In [27]:
pmb428 = read_sbml_model("/home/emma/Dokumente/thesis/Model_generation_curation/Curated_models/mb1/428_or_mb1.xml") #post mass balance

In [28]:
pmc428 = read_sbml_model("/home/emma/Dokumente/thesis/Model_generation_curation/Curated_models/mb1_mdr_rdr_dp_lib/428_or_mb1_mdr_rdr_dp_lib.xml")

In [18]:
sol2 = pmb428.optimize()
sol2.objective_value #problem is post mass balance

0.0

In [25]:
# get all biomass precursors
biomass_prec_428 = [met.id for met in pmc428.reactions.Growth.reactants]

In [ ]:
store_dict_428, demand_all_428 = test_biomass_prec(pmb428)

Added Reaction:  --> 5.0 10fthf_c
Optimization Status: optimal
Objective Value: 0.0
Added Reaction:  --> 5.0 ala__L_c
Optimization Status: optimal
Objective Value: 0.0
Added Reaction:  --> 5.0 amet_c
Optimization Status: optimal
Objective Value: 0.0
Added Reaction:  --> 5.0 arg__L_c
Optimization Status: optimal
Objective Value: 0.0
Added Reaction:  --> 5.0 asn__L_c
Optimization Status: optimal
Objective Value: 0.0
Added Reaction:  --> 5.0 asp__L_c
Optimization Status: optimal
Objective Value: 0.0
Added Reaction:  --> 5.0 atp_c
Optimization Status: optimal
Objective Value: 0.0
Added Reaction:  --> 5.0 ca2_c
Optimization Status: optimal
Objective Value: 0.0
Added Reaction:  --> 5.0 cl_c
Optimization Status: optimal
Objective Value: 0.0
Added Reaction:  --> 5.0 coa_c
Optimization Status: optimal
Objective Value: 0.0
Added Reaction:  --> 5.0 cobalt2_c
Optimization Status: optimal
Objective Value: 0.0
Added Reaction:  --> 5.0 ctp_c
Optimization Status: optimal
Objective Value: 0.0
Added Rea

In [ ]:
atp_syn = Reaction("ATP_artifical")

atp_syn.add_metabolites({
    pmb428.metabolites.atp_c: 4.0
})

In [ ]:
with pmb428 as model:
    model.add_reactions([atp_syn])
    print(model.reactions.get_by_id("ATP_artifical"))
    sol = model.optimize()
    print(model.objective)
    met_0flux_summary = []
    for met in gmets:
        prec_d, met_l = get_producing_reactions(model, [met])
        demand_biom = demand_reaction(model, met_l)
        m0f = [met_id for met_id, flux in demand_biom.items() if flux == 0.0] #only get fluxes that are 0 for further downstream ana
        if len(m0f) != 0:
            met_0flux_summary.append(m0f)

ATP_artifical:  --> 4.0 atp_c
Maximize
1.0*Growth - 1.0*Growth_reverse_699ae
=================== Analyzing: 10fthf_c ===================
Reaction: MTHFC
Precursors needed: h2o_c, methf_c
Max production flux of h2o_c     : 1000.0000
Max production flux of methf_c   : 0.0000
=================== Analyzing: ala__L_c ===================
Reaction: CYSDSF
Precursors needed: cys__L_c
Reaction: ALAt4
Precursors needed: ala__L_e, na1_e
Max production flux of cys__L_c  : 438.2022
Max production flux of ala__L_e  : 1000.0000
Max production flux of na1_e     : 1000.0000
=================== Analyzing: amet_c ===================
Reaction: MHPGLUT2
Precursors needed: ahcys_c, h_c, mhpglu_c
Reaction: METAT
Precursors needed: atp_c, h2o_c, met__L_c
Max production flux of ahcys_c   : 0.0000
Max production flux of h_c       : 1000.0000
Max production flux of mhpglu_c  : 0.0000
Max production flux of atp_c     : 1000.0000
Max production flux of h2o_c     : 1000.0000
Max production flux of met__L_c  : 1000.

#### m_1338

In [ ]:
pc1338 = read_sbml_model("/home/emma/Dokumente/thesis/Model_generation_curation/Draft_models/Models/1338.xml")

Adding exchange reaction EX_2obut_e with default bounds for boundary metabolite: 2obut_e.
Adding exchange reaction EX_6pgc_e with default bounds for boundary metabolite: 6pgc_e.
Adding exchange reaction EX_LalaDgluMdap_e with default bounds for boundary metabolite: LalaDgluMdap_e.
Adding exchange reaction EX_R_3httdca_e with default bounds for boundary metabolite: R_3httdca_e.
Adding exchange reaction EX_ac_e with default bounds for boundary metabolite: ac_e.
Adding exchange reaction EX_acald_e with default bounds for boundary metabolite: acald_e.
Adding exchange reaction EX_acmana_e with default bounds for boundary metabolite: acmana_e.
Adding exchange reaction EX_acnam_e with default bounds for boundary metabolite: acnam_e.
Adding exchange reaction EX_akg_e with default bounds for boundary metabolite: akg_e.
Adding exchange reaction EX_ala__L_e with default bounds for boundary metabolite: ala__L_e.
Adding exchange reaction EX_alaala_e with default bounds for boundary metabolite: alaa

In [ ]:
sol1 = pc1338.slim_optimize()
print(sol1)

16.62927698456265


In [ ]:
pmb1338 = read_sbml_model("/home/emma/Dokumente/thesis/Model_generation_curation/Curated_models/mb1/1338_or_mb1.xml") #post mass balance

In [ ]:
sol2 = pmb1338.slim_optimize()
print(sol2)

16.62927698456265


In [ ]:
pmd1338 = read_sbml_model("/home/emma/Dokumente/thesis/Model_generation_curation/Curated_models/mb1_mdr/1338_or_mb1_mdr.xml")

In [ ]:
sol3 = pmd1338.slim_optimize()
print(sol3)

16.62927698456265


In [ ]:
pmac1338 = read_sbml_model("/home/emma/Dokumente/thesis/Model_generation_curation/Curated_models/mb1_mdr_rdr_dp/1338_or_mb1_mdr_rdr_dp.xml")

In [ ]:
sol4 = pmac1338.slim_optimize()
print(sol4)

0.0


In [ ]:
store_dict_1338, demand_all_1338 = test_biomass_prec(pmb1338)

Added Reaction:  --> 5.0 10fthf_c
Optimization Status: optimal
Objective Value: 16.629338651617566
Added Reaction:  --> 5.0 ala__L_c
Optimization Status: optimal
Objective Value: 16.77255277070705
Added Reaction:  --> 5.0 amet_c
Optimization Status: optimal
Objective Value: 16.629379763241587
Added Reaction:  --> 5.0 arg__L_c
Optimization Status: optimal
Objective Value: 16.629276984562647
Added Reaction:  --> 5.0 asn__L_c
Optimization Status: optimal
Objective Value: 16.64905986962804
Added Reaction:  --> 5.0 asp__L_c
Optimization Status: optimal
Objective Value: 16.629276984562647
Added Reaction:  --> 5.0 atp_c
Optimization Status: optimal
Objective Value: 16.677668878498956
Added Reaction:  --> 5.0 ca2_c
Optimization Status: optimal
Objective Value: 16.63071646265725
Added Reaction:  --> 5.0 cl_c
Optimization Status: optimal
Objective Value: 16.629276984562647
Added Reaction:  --> 5.0 coa_c
Optimization Status: optimal
Objective Value: 16.62927698456265
Added Reaction:  --> 5.0 coba

#### m_892

In [ ]:
pc892 = read_sbml_model("/home/emma/Dokumente/thesis/Model_generation_curation/Draft_models/Models/892.xml")

Adding exchange reaction EX_2obut_e with default bounds for boundary metabolite: 2obut_e.
Adding exchange reaction EX_2pg_e with default bounds for boundary metabolite: 2pg_e.
Adding exchange reaction EX_3ump_e with default bounds for boundary metabolite: 3ump_e.
Adding exchange reaction EX_6pgc_e with default bounds for boundary metabolite: 6pgc_e.
Adding exchange reaction EX_LalaDgluMdap_e with default bounds for boundary metabolite: LalaDgluMdap_e.
Adding exchange reaction EX_R_3httdca_e with default bounds for boundary metabolite: R_3httdca_e.
Adding exchange reaction EX_acald_e with default bounds for boundary metabolite: acald_e.
Adding exchange reaction EX_acgam1p_e with default bounds for boundary metabolite: acgam1p_e.
Adding exchange reaction EX_acnam_e with default bounds for boundary metabolite: acnam_e.
Adding exchange reaction EX_ala__L_e with default bounds for boundary metabolite: ala__L_e.
Adding exchange reaction EX_alaala_e with default bounds for boundary metabolite

In [ ]:
sol1 = pc892.slim_optimize()
print(sol1)

16.070808496501453


In [ ]:
pmb892 = read_sbml_model("/home/emma/Dokumente/thesis/Model_generation_curation/Curated_models/mb1/892_or_mb1.xml") #post mass balance

In [ ]:
sol2 = pmb892.slim_optimize()
print(sol2)

0.0


In [ ]:
store_dict_892, demand_all_892 = test_biomass_prec(pmb892)

Added Reaction:  --> 5.0 10fthf_c
Optimization Status: optimal
Objective Value: 0.0
Added Reaction:  --> 5.0 ala__L_c
Optimization Status: optimal
Objective Value: 0.0
Added Reaction:  --> 5.0 amet_c
Optimization Status: optimal
Objective Value: 0.0
Added Reaction:  --> 5.0 arg__L_c
Optimization Status: optimal
Objective Value: 0.0
Added Reaction:  --> 5.0 asn__L_c
Optimization Status: optimal
Objective Value: 0.0
Added Reaction:  --> 5.0 asp__L_c
Optimization Status: optimal
Objective Value: 0.0
Added Reaction:  --> 5.0 atp_c
Optimization Status: optimal
Objective Value: 0.0
Added Reaction:  --> 5.0 ca2_c
Optimization Status: optimal
Objective Value: 0.0
Added Reaction:  --> 5.0 cl_c
Optimization Status: optimal
Objective Value: 0.0
Added Reaction:  --> 5.0 coa_c
Optimization Status: optimal
Objective Value: 0.0
Added Reaction:  --> 5.0 cobalt2_c
Optimization Status: optimal
Objective Value: 0.0
Added Reaction:  --> 5.0 ctp_c
Optimization Status: optimal
Objective Value: 0.0
Added Rea

#### m_1101

In [ ]:
pc1101 = read_sbml_model("/home/emma/Dokumente/thesis/Model_generation_curation/Draft_models/Models/1101.xml")

Adding exchange reaction EX_6pgc_e with default bounds for boundary metabolite: 6pgc_e.
Adding exchange reaction EX_LalaDgluMdap_e with default bounds for boundary metabolite: LalaDgluMdap_e.
Adding exchange reaction EX_R_3httdca_e with default bounds for boundary metabolite: R_3httdca_e.
Adding exchange reaction EX_ac_e with default bounds for boundary metabolite: ac_e.
Adding exchange reaction EX_acgam1p_e with default bounds for boundary metabolite: acgam1p_e.
Adding exchange reaction EX_adn_e with default bounds for boundary metabolite: adn_e.
Adding exchange reaction EX_akg_e with default bounds for boundary metabolite: akg_e.
Adding exchange reaction EX_ala_L_asp__L_e with default bounds for boundary metabolite: ala_L_asp__L_e.
Adding exchange reaction EX_ala__D_e with default bounds for boundary metabolite: ala__D_e.
Adding exchange reaction EX_ala_leu_e with default bounds for boundary metabolite: ala_leu_e.
Adding exchange reaction EX_arg__L_e with default bounds for boundary 

In [ ]:
pc1101.metabolites.fdxo_2_2_c

Metabolite identifier,fdxo_2_2_c
Name,Oxidized ferredoxin
Memory address,0x752d29bd3be0
Formula,Fe2S2
Compartment,C_c
In 2 reaction(s),"MECDPDH4E, POR_syn"


In [ ]:
pmb1101.metabolites.fdxox_c

Metabolite identifier,fdxox_c
Name,Oxidized ferredoxin
Memory address,0x752d2b297310
Formula,Fe2S2
Compartment,c
In 1 reaction(s),MECDPDH3_syn


In [ ]:
sol1 = pc1101.slim_optimize()
print(sol1)

16.421548423560736


In [ ]:
pmb1101 = read_sbml_model("/home/emma/Dokumente/thesis/Model_generation_curation/Curated_models/mb1/1101_or_mb1.xml") #post mass balance

In [ ]:
sol2 = pmb1101.slim_optimize()
print(sol2)

0.0


In [ ]:
store_dict_1101, demand_all_1101 = test_biomass_prec(pmb1101)

Added Reaction:  --> 5.0 10fthf_c
Optimization Status: optimal
Objective Value: 0.0
Added Reaction:  --> 5.0 ala__L_c
Optimization Status: optimal
Objective Value: 0.0
Added Reaction:  --> 5.0 amet_c
Optimization Status: optimal
Objective Value: 0.0
Added Reaction:  --> 5.0 arg__L_c
Optimization Status: optimal
Objective Value: 0.0
Added Reaction:  --> 5.0 asn__L_c
Optimization Status: optimal
Objective Value: 0.0
Added Reaction:  --> 5.0 asp__L_c
Optimization Status: optimal
Objective Value: 0.0
Added Reaction:  --> 5.0 atp_c
Optimization Status: optimal
Objective Value: 0.0
Added Reaction:  --> 5.0 ca2_c
Optimization Status: optimal
Objective Value: 0.0
Added Reaction:  --> 5.0 cl_c
Optimization Status: optimal
Objective Value: 0.0
Added Reaction:  --> 5.0 coa_c
Optimization Status: optimal
Objective Value: 0.0
Added Reaction:  --> 5.0 cobalt2_c
Optimization Status: optimal
Objective Value: 0.0
Added Reaction:  --> 5.0 ctp_c
Optimization Status: optimal
Objective Value: 0.0
Added Rea

In [ ]:
store_dict_1101["mql8_c"]

[[], 'optimal', 16.421631892420343]

#### m_644


In [ ]:
pc644 = read_sbml_model("/home/emma/Dokumente/thesis/Model_generation_curation/Draft_models/Models/644.xml")

Adding exchange reaction EX_6apa_e with default bounds for boundary metabolite: 6apa_e.
Adding exchange reaction EX_6pgc_e with default bounds for boundary metabolite: 6pgc_e.
Adding exchange reaction EX_LalaDgluMdap_e with default bounds for boundary metabolite: LalaDgluMdap_e.
Adding exchange reaction EX_R_3httdca_e with default bounds for boundary metabolite: R_3httdca_e.
Adding exchange reaction EX_ac_e with default bounds for boundary metabolite: ac_e.
Adding exchange reaction EX_acald_e with default bounds for boundary metabolite: acald_e.
Adding exchange reaction EX_acmana_e with default bounds for boundary metabolite: acmana_e.
Adding exchange reaction EX_acnam_e with default bounds for boundary metabolite: acnam_e.
Adding exchange reaction EX_ala__L_e with default bounds for boundary metabolite: ala__L_e.
Adding exchange reaction EX_alaala_e with default bounds for boundary metabolite: alaala_e.
Adding exchange reaction EX_amp_e with default bounds for boundary metabolite: amp

In [ ]:
sol1 = pc644.slim_optimize()
print(sol1)

16.8765851142707


In [ ]:
pmb644 = read_sbml_model("/home/emma/Dokumente/thesis/Model_generation_curation/Curated_models/mb1/644_or_mb1.xml") #post mass balance

In [ ]:
sol2 = pmb644.slim_optimize()
print(sol2)

0.0


In [ ]:
store_dict_644, demand_all_644 = test_biomass_prec(pmb644)

Added Reaction:  --> 5.0 10fthf_c
Optimization Status: optimal
Objective Value: 0.0
Added Reaction:  --> 5.0 ala__L_c
Optimization Status: optimal
Objective Value: 0.0
Added Reaction:  --> 5.0 amet_c
Optimization Status: optimal
Objective Value: 0.0
Added Reaction:  --> 5.0 arg__L_c
Optimization Status: optimal
Objective Value: 0.0
Added Reaction:  --> 5.0 asn__L_c
Optimization Status: optimal
Objective Value: 0.0
Added Reaction:  --> 5.0 asp__L_c
Optimization Status: optimal
Objective Value: 0.0
Added Reaction:  --> 5.0 atp_c
Optimization Status: optimal
Objective Value: 0.0
Added Reaction:  --> 5.0 ca2_c
Optimization Status: optimal
Objective Value: 0.0
Added Reaction:  --> 5.0 cl_c
Optimization Status: optimal
Objective Value: 0.0
Added Reaction:  --> 5.0 coa_c
Optimization Status: optimal
Objective Value: 0.0
Added Reaction:  --> 5.0 cobalt2_c
Optimization Status: optimal
Objective Value: 0.0
Added Reaction:  --> 5.0 ctp_c
Optimization Status: optimal
Objective Value: 0.0
Added Rea

#### m_1056

In [ ]:
pc1056 = read_sbml_model("/home/emma/Dokumente/thesis/Model_generation_curation/Draft_models/Models/1056.xml")

Adding exchange reaction EX_2obut_e with default bounds for boundary metabolite: 2obut_e.
Adding exchange reaction EX_6pgc_e with default bounds for boundary metabolite: 6pgc_e.
Adding exchange reaction EX_LalaDgluMdap_e with default bounds for boundary metabolite: LalaDgluMdap_e.
Adding exchange reaction EX_R_3httdca_e with default bounds for boundary metabolite: R_3httdca_e.
Adding exchange reaction EX_acald_e with default bounds for boundary metabolite: acald_e.
Adding exchange reaction EX_acmana_e with default bounds for boundary metabolite: acmana_e.
Adding exchange reaction EX_acnam_e with default bounds for boundary metabolite: acnam_e.
Adding exchange reaction EX_adn_e with default bounds for boundary metabolite: adn_e.
Adding exchange reaction EX_arg__L_e with default bounds for boundary metabolite: arg__L_e.
Adding exchange reaction EX_asn__L_e with default bounds for boundary metabolite: asn__L_e.
Adding exchange reaction EX_asp__L_e with default bounds for boundary metaboli

In [ ]:
sol1 = pc1056.slim_optimize()
print(sol1)

16.742134488712495


In [ ]:
pmb1056 = read_sbml_model("/home/emma/Dokumente/thesis/Model_generation_curation/Curated_models/mb1/1056_or_mb1.xml") #post mass balance

In [ ]:
sol2 = pmb1056.slim_optimize()
print(sol2)

0.0


In [ ]:
store_dict_1056, demand_all_1056 = test_biomass_prec(pmb1056)

Added Reaction:  --> 5.0 10fthf_c
Optimization Status: optimal
Objective Value: 0.0
Added Reaction:  --> 5.0 ala__L_c
Optimization Status: optimal
Objective Value: 0.0
Added Reaction:  --> 5.0 amet_c
Optimization Status: optimal
Objective Value: 0.0
Added Reaction:  --> 5.0 arg__L_c
Optimization Status: optimal
Objective Value: 0.0
Added Reaction:  --> 5.0 asn__L_c
Optimization Status: optimal
Objective Value: 0.0
Added Reaction:  --> 5.0 asp__L_c
Optimization Status: optimal
Objective Value: 0.0
Added Reaction:  --> 5.0 atp_c
Optimization Status: optimal
Objective Value: 0.0
Added Reaction:  --> 5.0 ca2_c
Optimization Status: optimal
Objective Value: 0.0
Added Reaction:  --> 5.0 cl_c
Optimization Status: optimal
Objective Value: 0.0
Added Reaction:  --> 5.0 coa_c
Optimization Status: optimal
Objective Value: 0.0
Added Reaction:  --> 5.0 cobalt2_c
Optimization Status: optimal
Objective Value: 0.0
Added Reaction:  --> 5.0 ctp_c
Optimization Status: optimal
Objective Value: 0.0
Added Rea

#### m_352

In [ ]:
pc352 = read_sbml_model("/home/emma/Dokumente/thesis/Model_generation_curation/Draft_models/Models/352.xml")

Adding exchange reaction EX_2obut_e with default bounds for boundary metabolite: 2obut_e.
Adding exchange reaction EX_6pgc_e with default bounds for boundary metabolite: 6pgc_e.
Adding exchange reaction EX_LalaDgluMdap_e with default bounds for boundary metabolite: LalaDgluMdap_e.
Adding exchange reaction EX_R_3httdca_e with default bounds for boundary metabolite: R_3httdca_e.
Adding exchange reaction EX_ac_e with default bounds for boundary metabolite: ac_e.
Adding exchange reaction EX_acgam1p_e with default bounds for boundary metabolite: acgam1p_e.
Adding exchange reaction EX_acnam_e with default bounds for boundary metabolite: acnam_e.
Adding exchange reaction EX_ala__L_e with default bounds for boundary metabolite: ala__L_e.
Adding exchange reaction EX_alaala_e with default bounds for boundary metabolite: alaala_e.
Adding exchange reaction EX_amp_e with default bounds for boundary metabolite: amp_e.
Adding exchange reaction EX_arg__L_e with default bounds for boundary metabolite: 

In [ ]:
sol1 = pc352.slim_optimize()
print(sol1)

15.985283308774612


In [ ]:
pmb352 = read_sbml_model("/home/emma/Dokumente/thesis/Model_generation_curation/Curated_models/mb1/352_or_mb1.xml") #post mass balance

In [ ]:
sol2 = pmb352.slim_optimize()
print(sol2)

15.985283308774612


In [ ]:
pmd352 = read_sbml_model("/home/emma/Dokumente/thesis/Model_generation_curation/Curated_models/mb1_mdr/352_or_mb1_mdr.xml")

In [ ]:
sol3 = pmd352.slim_optimize()
print(sol3)

15.985283308774612


In [ ]:
pmac352 = read_sbml_model("/home/emma/Dokumente/thesis/Model_generation_curation/Curated_models/mb1_mdr_rdr_dp/352_or_mb1_mdr_rdr_dp.xml")

In [ ]:
sol4 = pmac352.slim_optimize()
print(sol4)

0.0


#### m_1432

In [ ]:
pc1432 = read_sbml_model("/home/emma/Dokumente/thesis/Model_generation_curation/Draft_models/Models/1432.xml")
sol1 = pc1432.slim_optimize()
print(sol1)
pmb1432 = read_sbml_model("/home/emma/Dokumente/thesis/Model_generation_curation/Curated_models/mb1/1432_or_mb1.xml") #post mass balance
sol2 = pmb1432.slim_optimize()
print(sol2)

Adding exchange reaction EX_5oxpro_e with default bounds for boundary metabolite: 5oxpro_e.
Adding exchange reaction EX_6pgc_e with default bounds for boundary metabolite: 6pgc_e.
Adding exchange reaction EX_LalaDgluMdap_e with default bounds for boundary metabolite: LalaDgluMdap_e.
Adding exchange reaction EX_R_3httdca_e with default bounds for boundary metabolite: R_3httdca_e.
Adding exchange reaction EX_ac_e with default bounds for boundary metabolite: ac_e.
Adding exchange reaction EX_acgam1p_e with default bounds for boundary metabolite: acgam1p_e.
Adding exchange reaction EX_ala__L_e with default bounds for boundary metabolite: ala__L_e.
Adding exchange reaction EX_alaala_e with default bounds for boundary metabolite: alaala_e.
Adding exchange reaction EX_amp_e with default bounds for boundary metabolite: amp_e.
Adding exchange reaction EX_arg__L_e with default bounds for boundary metabolite: arg__L_e.
Adding exchange reaction EX_asn__L_e with default bounds for boundary metaboli

15.278167981127032
15.281848623472191


In [ ]:
pmd1432 = read_sbml_model("/home/emma/Dokumente/thesis/Model_generation_curation/Curated_models/mb1_mdr/1432_or_mb1_mdr.xml")
sol3 = pmd1432.slim_optimize()
print(sol3)
pmac1432 = read_sbml_model("/home/emma/Dokumente/thesis/Model_generation_curation/Curated_models/mb1_mdr_rdr_dp/1432_or_mb1_mdr_rdr_dp.xml")
sol4 = pmac1432.slim_optimize()
print(sol4)

15.281848623472191
0.0


### fix biomass precursors fe2 and mql8

#### fe2

- copy ABC transporter from Fe3+ and adjust for Fe2+

In [ ]:
# create all required fe2 metabolites
fe2_e = Metabolite(id = "fe2_e", name = "Iron (Fe2+)", formula="Fe", compartment="C_e")
fe2_p = Metabolite(id = "fe2_p", name = "Iron (Fe2+)", formula="Fe", compartment="C_p")

In [114]:
# exchange reaction EX_fe2_e
EX_fe2_e = Reaction("EX_fe2_e")
EX_fe2_e.name = f"EX_fe2_e"
EX_fe2_e.lower_bound = -1000.0 
EX_fe2_e.upper_bound = 1000.0  
EX_fe2_e.add_metabolites({fe2_e: -1.0})

In [116]:
# fe2_e -> fe2_p FE2tex
FE2tex = Reaction("FE2tex")
FE2tex.name = f"FE2tex"
FE2tex.lower_bound = -1000.0 
FE2tex.upper_bound = 1000.0  
FE2tex.add_metabolites({fe2_e: -1.0, fe2_p: 1.0})

In [122]:
# fe2_p -> fe2_c FE2abcpp
FE2abcpp = Reaction("FE2abcpp")
FE2abcpp.name = f"FE2abcpp"
FE2abcpp.lower_bound = -1000.0 
FE2abcpp.upper_bound = 1000.0  
FE2abcpp.add_metabolites({fe2_p: -1.0, 
                          model.metabolites.get_by_id("atp_c"): -1.0, 
                          model.metabolites.get_by_id("h2o_c"): -1.0, 
                          model.metabolites.get_by_id("adp_c"): 1.0, 
                          model.metabolites.get_by_id("h_c"): 1.0, 
                          model.metabolites.get_by_id("pi_c"): 1.0,
                        model.metabolites.get_by_id("fe2_c"): 1.0})


In [124]:
fe2_models = [pmb428, pmb644]

In [125]:
for model in fe2_models:
    with model:
        sol1 = model.optimize()
        print(sol1.objective_value)
        model.add_metabolites([fe2_e, fe2_p])
        model.add_reactions([EX_fe2_e, FE2tex, FE2abcpp])
        sol2 = model.optimize()
        print(sol2.objective_value)


0.0
16.8765851142707
0.0
16.8765851142707


#### mql8

- problem was that POR_syn was empty as fdxox was never properly defined (defined as "" but should be metabolite object)
- replace mql8 with q8n2 and add synthesis pathways, 1101 can not produce mql8
- replace in growth reacitons as well 

In [112]:
# manually add fdxox_c as a metabolite
fdxox_c = Metabolite(
    'fdxox_c',
    formula='Fe2S2',
    name='Oxidized ferredoxin',
    compartment='c')

In [113]:
POR_new_rxn_dict = {fdxox_c: -2.0, # replaces fdxo_2_2_c
                        "coa_c": -1.0,
                        "pyr_c": -1.0,
                        "accoa_c": 1.0,
                        "co2_c": 1.0,
                        "h_c": 1.0,
                        "fdxrd_c": 2.0}

In [209]:
with pmb1101 as model:
    #rxn = model.reactions.POR_syn
    print(model.reactions.POR_syn)
    #model.reactions.POR_syn.subtract_metabolites(model.reactions.POR_syn.metabolites)
    model.reactions.POR_syn.add_metabolites(POR_new_rxn_dict)
    print(model.reactions.POR_syn)
    #model.add_reactions([model.reactions.POR_syn])
    sol = model.slim_optimize()
    print(sol)
    store_dict_1101, demand_all_1101 = test_biomass_prec(model)
    
    

POR_syn:  --> 
POR_syn: coa_c + 2.0 fdxox_c + pyr_c --> accoa_c + co2_c + 2.0 fdxrd_c + h_c
0.0
Added Reaction:  --> 5.0 10fthf_c
Optimization Status: optimal
Objective Value: 0.0
Added Reaction:  --> 5.0 ala__L_c
Optimization Status: optimal
Objective Value: 0.0
Added Reaction:  --> 5.0 amet_c
Optimization Status: optimal
Objective Value: 0.0
Added Reaction:  --> 5.0 arg__L_c
Optimization Status: optimal
Objective Value: 0.0
Added Reaction:  --> 5.0 asn__L_c
Optimization Status: optimal
Objective Value: 0.0
Added Reaction:  --> 5.0 asp__L_c
Optimization Status: optimal
Objective Value: 0.0
Added Reaction:  --> 5.0 atp_c
Optimization Status: optimal
Objective Value: 0.0
Added Reaction:  --> 5.0 ca2_c
Optimization Status: optimal
Objective Value: 0.0
Added Reaction:  --> 5.0 cl_c
Optimization Status: optimal
Objective Value: 0.0
Added Reaction:  --> 5.0 coa_c
Optimization Status: optimal
Objective Value: 0.0
Added Reaction:  --> 5.0 cobalt2_c
Optimization Status: optimal
Objective Value

##### replace with q8h2

In [20]:
# manually add q8h2_c and other required metabolites
q8h2_c = Metabolite(
    'q8h2_c',
    formula='C49H76O4', name='Ubiquinol-8', compartment='c')

ohph_c = Metabolite(
    '2ohph_c',
    formula='C46H70O2', name='2-Octaprenyl-6-hydroxyphenol',compartment='c')

omph_c = Metabolite(
    '2omph_c',
    formula='C47H72O2', name='2-Octaprenyl-6-methoxyphenol',compartment='c')

ombzl_c = Metabolite(
    '2ombzl_c',
    formula='C47H72O3', name='2-Octaprenyl-6-methoxy-1,4-benzoquinol', compartment='c')

ommbl_c = Metabolite(
    '2ommbl_c',
    formula='C48H74O3', name='2-Octaprenyl-3-methyl-6-methoxy- 1,4-benzoquinol', compartment='c')

omhmbl_c = Metabolite(
    '2omhmbl_c',
    formula='C48H74O4', name='2-Octaprenyl-3-methyl-5-hydroxy-6-methoxy-1,4-benzoquinol', compartment='c')

In [ ]:
# taken from Lisa 
def add_new_rxn(model, id, name, lb, ub, stoich):
    if id not in model.reactions:
        new_rxn = Reaction(id=id, name=name, lower_bound=lb, upper_bound=ub)

        met_objs = {}
        for met_id, coeff in stoich.items():
            met_obj = model.metabolites.get_by_id(met_id) if met_id in model.metabolites else None
            if met_obj is None:
                return f"{met_id} not in {model.id}; {id} was not added"
            met_objs[met_obj] = coeff

        new_rxn.add_metabolites(met_objs)
        model.add_reactions([new_rxn])
        return None
    else:
        return f"{id} already in {model.id}"

In [64]:
def add_new_met(model, id, name, formula, charge, compartment):
    if id not in model.metabolites:
        new_met = Metabolite(id, name=name, formula=formula, charge=charge, compartment=compartment)
        model.add_metabolites([new_met])
        return None
    else:
        return f"{id} already in {model.id}"



In [76]:
def add_ubi_metabolites(model):    
    add_new_met(model, "4hbz_e", '4-Hydroxybenzoate', 'C7H5O3', -1, "C_e")
    add_new_met(model, "4hbz_p", '4-Hydroxybenzoate', 'C7H5O3', -1, "C_p")
    add_new_met(model, "4hbz_c", '4-Hydroxybenzoate', 'C7H5O3', -1, "C_c")
    add_new_met(model, "3ophb_c", '3-Octaprenyl-4-hydroxybenzoate', 'C47H69O3', -1, "C_c")
    add_new_met(model, "2oph_c", '2-Octaprenylphenol', 'C46H70O', 0, "C_c")
    add_new_met(model, "2ohph_c", '2-Octaprenyl-6-hydroxyphenol', 'C46H70O2', 0, "C_c") # Lisa only adds down from here
    add_new_met(model, "2omph_c",'2-Octaprenyl-6-methoxyphenol', 'C47H72O2', 0, "_c")
    add_new_met(model, "2ombzl_c",'2-Octaprenyl-6-methoxy-1,4-benzoquinol', 'C47H72O3', 0, "C_c")
    add_new_met(model, "2ommbl_c",'2-Octaprenyl-3-methyl-6-methoxy-1,4-benzoquinol', 'C48H74O3', 0, "C_c")
    add_new_met(model, "2omhmbl_c",'2-Octaprenyl-3-methyl-5-hydroxy-6-methoxy-1,4-benzoquinol', 'C48H74O4', 0, "C_c")
    add_new_met(model, "q8h2_c",'Ubiquinol-8', 'C49H76O4', 0, "C_c")


In [ ]:
# list of reactions that need to be added
# EX_4hbz_e: 4hbz_e ⇌ 
# 4HBZtex: 4hbz_e ⇌ 4hbz_p
# 4HBZt3pp: 4hbz_c + h_p ⇌ h_c + 4hbz_p
# HBZOPT: 4hbz_c + octdp_c ⇌ 3ophb_c + ppi_c
# OPHBDC: 3ophb_c + h_c ⇌ 2oph_c + co2_c
# OPHHX: 2oph_c + 0.5 o2_c ⇌ 2ohph_c 
# 2.1.1.222 = OHPHM: 2ohph_c + amet_c → 2omph_c + ahcys_c + h_c #lisa adds downstream from here
# 1.14.13.- = OMPHHX: 2omph_c + 0.5 o2_c → 2ombzl_c
# 2.1.1.201 = OMBZLM: 2ombzl_c + amet_c → 2ommbl_c + ahcys_c + h_c
# 1.14.99.60 = OMMBLHX: 2ommbl_c + 0.5 o2_c → 2omhmbl_c
# 2.1.1.64 = DMQMT: 2omhmbl_c + amet_c → ahcys_c + h_c + q8h2_c


In [ ]:
# taken from Lisa - # add synthesis reactions 2.1.1.222 (OHPHM), 1.14.13.- (OMPHHX), 2.1.1.201 (OMBLZM), 1.14.99.60 (OMMBLHX) and 2.1.1.64 (DMQMT), directionality as done by Lisa 
def add_EX_4hbz_e(model): #need to add EX_4hbz_e to medium!!!
    add_new_rxn(model, "EX_4hbz_e", "Exchange 4hbz", -1000, 1000,
                {"4hbz_e": 1.0})
    
def add_4HBZtex(model):
    add_new_rxn(model, "4HBZtex", "4-Hydroxybenzoate transport (extracellular)", 0, 1000,
                {"4hbz_e": -1.0, "4hbz_p": 1.0})
    
def add_4HBZt3pp(model):
    add_new_rxn(model, "4HBZt3pp", "4-hydroxybenzoate transport out via antiporter", 0, 1000,
                {"4hbz_p": -1.0, "h_p": -1.0, "4hbz_c": 1.0, "h_c": 1.0})
    
def add_HBZOPT(model):
    add_new_rxn(model, "HBZOPT", "Hydroxybenzoate octaprenyltransferase", 0, 1000,
                {"4hbz_c": -1.0, "octdp_c": -1.0, '3ophb_c': 1.0, "ppi_c": 1.0})
    
def add_OPHBDC(model):
    add_new_rxn(model, "OPHBDC", "Octaprenyl-hydroxybenzoate decarboxylase", 0, 1000,
                {"3ophb_c": -1.0, "h_c": -1.0, '2oph_c': 1.0, "co2_c": 1.0})
    
def add_OPHHX(model):
    add_new_rxn(model, "OPHHX", "2-Octaprenylphenol hydroxylase", 0, 1000,
            {'2oph_c': -1.0, "o2_c": -0.5, "2ohph_c": 1.0})
    
def add_OHPHM(model):
    add_new_rxn(model, "OHPHM", "2-octaprenyl-6-hydroxyphenol methylase", 0, 1000,
                {'2ohph_c': -1.0, '2omph_c': 1.0, 'ahcys_c': 1.0, 'amet_c': -1.0, 'h_c': 1.0})

def add_OMPHHX(model):
    add_new_rxn(model, "OMPHHX", "2-octaprenyl-6-methoxyphenol hydroxylase", 0, 1000,
                {'2ombzl_c': 1.0, '2omph_c': -1.0, 'o2_c': -0.5})

def add_OMBZLM(model):
    add_new_rxn(model, "OMBZLM", "2-Octaprenyl-6-methoxy-benzoquinol methylase", 0, 1000,
                {'2ombzl_c': -1.0, '2ommbl_c': 1.0, 'ahcys_c': 1.0, 'amet_c': -1.0, 'h_c': 1.0})

def add_OMMBLHX(model):
    add_new_rxn(model, "OMMBLHX", "2-Octaprenyl-3-methyl-6-methoxy-1,4-benzoquinol hydroxylase", 0, 1000,
                {'2omhmbl_c': 1.0, '2ommbl_c': -1.0, 'o2_c': -0.5})

def add_DMQMT(model):
    add_new_rxn(model, "DMQMT", "3-Dimethylubiquinonol 3-methyltransferase", 0, 1000,
                {'2omhmbl_c': -1.0, 'ahcys_c': 1.0, 'amet_c': -1.0, 'h_c': 1.0, 'q8h2_c': 1.0})

In [ ]:
with pmb1101 as model:
    model.reactions.POR_syn.add_metabolites(POR_new_rxn_dict)
    print(model.reactions.POR_syn)
    sol = model.slim_optimize()
    print(sol)
    add_ubi_metabolites(model)
    add_EX_4hbz_e(model)
    add_4HBZtex(model)
    add_4HBZt3pp(model)
    add_HBZOPT(model)
    add_OPHBDC(model)
    add_OPHHX(model)
    add_OHPHM(model)
    add_OMPHHX(model)
    add_OMBZLM(model)
    add_OMMBLHX(model)
    add_DMQMT(model)

    med = model.medium
    med["EX_4hbz_e"] = 1000.0
    model.medium = med
    print(model.medium)

    growth_rxn = model.reactions.get_by_id("Growth") #remove mql8 from growth reaction 
    met = model.metabolites.get_by_id("mql8_c")
    if met in growth_rxn.metabolites:
        growth_rxn.subtract_metabolites({met: growth_rxn.metabolites[met]})
    growth_rxn.add_metabolites({q8h2_c : -0.0001}) #stochiometry acc to Lisa
    print(growth_rxn)

    sol1 = model.slim_optimize()
    print(sol1)
    #store_dict_1101, demand_all_1101 = test_biomass_prec(model)
    
    dr = demand_reaction(model, ["frdp_c", "ipdp_c", "octdp_c", "dmpp_c", "h2mb4p_c", "2mecdp_c", "2p4c2me_c", "4c2me_c", 
                                 "2me4p_c", "dxyl5p_c", "g3p_c", "pyr_c", "uaccg_c", "pep_c", "ala__L_c", "ala__D_p"]) #problem: octdp_c is not produced 
    print(dr)

POR_syn: coa_c + 2.0 fdxox_c + pyr_c --> accoa_c + co2_c + 2.0 fdxrd_c + h_c
0.0
{'EX_6pgc_e': 1000.0, 'EX_LalaDgluMdap_e': 1000.0, 'EX_R_3httdca_e': 1000.0, 'EX_ac_e': 1000.0, 'EX_acgam1p_e': 1000.0, 'EX_adn_e': 1000.0, 'EX_akg_e': 1000.0, 'EX_ala_L_asp__L_e': 1000.0, 'EX_ala__D_e': 1000.0, 'EX_ala_leu_e': 1000.0, 'EX_arg__L_e': 1000.0, 'EX_asn__L_e': 1000.0, 'EX_bhb_e': 1000.0, 'EX_bz_e': 1000.0, 'EX_ca2_e': 1000.0, 'EX_cl_e': 1000.0, 'EX_cmp_e': 1000.0, 'EX_co2_e': 1000.0, 'EX_co_e': 1000.0, 'EX_coa_e': 1000.0, 'EX_cobalt2_e': 1000.0, 'EX_cu2_e': 1000.0, 'EX_dtmp_e': 1000.0, 'EX_fad_e': 1000.0, 'EX_fald_e': 1000.0, 'EX_fe2_e': 1000.0, 'EX_fe3_e': 1000.0, 'EX_fe3pyovd_kt_e': 1000.0, 'EX_fol_e': 1000.0, 'EX_g3pg_e': 1000.0, 'EX_gln__L_e': 1000.0, 'EX_gly_e': 1000.0, 'EX_gmp_e': 1000.0, 'EX_h2o_e': 1000.0, 'EX_h2s_e': 1000.0, 'EX_h_e': 1000.0, 'EX_hcys__L_e': 1000.0, 'EX_hdca_e': 1000.0, 'EX_his__L_e': 1000.0, 'EX_ile__L_e': 1000.0, 'EX_k_e': 1000.0, 'EX_leu__L_e': 1000.0, 'EX_lys__L_e

#### atp/adp production is blocked 

In [ ]:
model = pmb428
with model as temp_model: #can we produce ATP with unlimited ADP?
    # 1. Break the cyclic dependency by adding a free source of ADP
    adp_source = temp_model.add_boundary(temp_model.metabolites.adp_c, type="sink")
    adp_source.lower_bound = -1000  # Allow unlimited free ADP input
    
    # 2. Create a demand for ATP to see if it can phosphorylate that ADP
    atp_demand = temp_model.add_boundary(temp_model.metabolites.atp_c, type="demand")
    temp_model.objective = atp_demand
    
    sol = temp_model.optimize()
    print(f"ATP production flux with free ADP: {sol.objective_value}")

ATP production flux with free ADP: 121.875


In [334]:
with model as temp_model: #can we produce energy from raw NADH?
    # 1. Provide free cytoplasmic ADP and Phosphate
    temp_model.add_boundary(temp_model.metabolites.adp_c, type="sink").lower_bound = -1000
    temp_model.add_boundary(temp_model.metabolites.pi_c, type="sink").lower_bound = -1000
    
    # 2. Provide free NADH (the primary electron donor for respiration)
    try:
        temp_model.add_boundary(temp_model.metabolites.nadh_c, type="sink").lower_bound = -1000
    except:
        print("nadh_c not found, trying nadh_m or similar...")
        
    # 3. Request ATP production
    atp_demand = temp_model.add_boundary(temp_model.metabolites.atp_c, type="demand")
    temp_model.objective = atp_demand
    
    print(f"ATP from raw NADH: {temp_model.optimize().objective_value}")

ATP from raw NADH: 600.0000000000002


In [335]:
with model as temp_model: # is atp synthase or respiratory chain broken? REsult: ATP Flux with direct proton power: 250.0 - ATP synthase works!
    # 1. Provide free ADP and Pi in the cytoplasm
    temp_model.add_boundary(temp_model.metabolites.adp_c, type="sink").lower_bound = -1000
    temp_model.add_boundary(temp_model.metabolites.pi_c, type="sink").lower_bound = -1000
    
    # 2. Provide a free source of PERIPLASMIC/EXTRACELLULAR protons 
    # (Check if your model uses h_p or h_e for the space outside the cytoplasm)
    try:
        temp_model.add_boundary(temp_model.metabolites.h_p, type="sink").lower_bound = -1000
    except KeyError:
        temp_model.add_boundary(temp_model.metabolites.h_e, type="sink").lower_bound = -1000

    # 3. Request ATP production
    temp_model.objective = temp_model.add_boundary(temp_model.metabolites.atp_c, type="demand")
    
    print(f"ATP Flux with direct proton power: {temp_model.optimize().objective_value}")

ATP Flux with direct proton power: 694.9152542372881


In [181]:
# need to check if metabolites in electron transport chain can be produced
resp_chain_mets = ["nadh_c", "nadp_c", "nadph_c", "fad_c", "fadh2_c"]


### post macaw probs

- all mods stop growing after deletion of FPRA 
    - this is a duplicate but gets deleted in all models, wo check if duplicate is there 

### check draft models

In [187]:
draft_model_dir = "/home/emma/Dokumente/thesis/Model_generation_curation/Draft_models/Models/"

In [191]:
all_blocked = {}

for file in os.listdir(draft_model_dir):
    if file.endswith('.xml'):
        model = read_sbml_model(os.path.join(draft_model_dir,file))
        blocked_components, blocked_reactions = check_growth(model)
        all_blocked.update(blocked_reactions)

Adding exchange reaction EX_3ump_e with default bounds for boundary metabolite: 3ump_e.
Adding exchange reaction EX_6pgc_e with default bounds for boundary metabolite: 6pgc_e.
Adding exchange reaction EX_LalaDgluMdap_e with default bounds for boundary metabolite: LalaDgluMdap_e.
Adding exchange reaction EX_R_3httdca_e with default bounds for boundary metabolite: R_3httdca_e.
Adding exchange reaction EX_ac_e with default bounds for boundary metabolite: ac_e.
Adding exchange reaction EX_acgam1p_e with default bounds for boundary metabolite: acgam1p_e.
Adding exchange reaction EX_acnam_e with default bounds for boundary metabolite: acnam_e.
Adding exchange reaction EX_ala_leu_e with default bounds for boundary metabolite: ala_leu_e.
Adding exchange reaction EX_alaala_e with default bounds for boundary metabolite: alaala_e.
Adding exchange reaction EX_amp_e with default bounds for boundary metabolite: amp_e.
Adding exchange reaction EX_arg__L_e with default bounds for boundary metabolite: 

Found 0 biomass components that cannot be synthesized for model m_2862.


Adding exchange reaction EX_5oxpro_e with default bounds for boundary metabolite: 5oxpro_e.
Adding exchange reaction EX_6pgc_e with default bounds for boundary metabolite: 6pgc_e.
Adding exchange reaction EX_LalaDgluMdap_e with default bounds for boundary metabolite: LalaDgluMdap_e.
Adding exchange reaction EX_R_3httdca_e with default bounds for boundary metabolite: R_3httdca_e.
Adding exchange reaction EX_ac_e with default bounds for boundary metabolite: ac_e.
Adding exchange reaction EX_acald_e with default bounds for boundary metabolite: acald_e.
Adding exchange reaction EX_acgam1p_e with default bounds for boundary metabolite: acgam1p_e.
Adding exchange reaction EX_acnam_e with default bounds for boundary metabolite: acnam_e.
Adding exchange reaction EX_ala_L_thr__L_e with default bounds for boundary metabolite: ala_L_thr__L_e.
Adding exchange reaction EX_alaala_e with default bounds for boundary metabolite: alaala_e.
Adding exchange reaction EX_amp_e with default bounds for bounda

Found 0 biomass components that cannot be synthesized for model m_796.


Adding exchange reaction EX_LalaDgluMdap_e with default bounds for boundary metabolite: LalaDgluMdap_e.
Adding exchange reaction EX_ac_e with default bounds for boundary metabolite: ac_e.
Adding exchange reaction EX_acald_e with default bounds for boundary metabolite: acald_e.
Adding exchange reaction EX_acgam1p_e with default bounds for boundary metabolite: acgam1p_e.
Adding exchange reaction EX_acnam_e with default bounds for boundary metabolite: acnam_e.
Adding exchange reaction EX_akg_e with default bounds for boundary metabolite: akg_e.
Adding exchange reaction EX_alaala_e with default bounds for boundary metabolite: alaala_e.
Adding exchange reaction EX_amp_e with default bounds for boundary metabolite: amp_e.
Adding exchange reaction EX_arg__L_e with default bounds for boundary metabolite: arg__L_e.
Adding exchange reaction EX_argp_e with default bounds for boundary metabolite: argp_e.
Adding exchange reaction EX_asn__L_e with default bounds for boundary metabolite: asn__L_e.
Ad

...

Found 0 biomass components that cannot be synthesized for model m_761.


Adding exchange reaction EX_2obut_e with default bounds for boundary metabolite: 2obut_e.
Adding exchange reaction EX_3amp_e with default bounds for boundary metabolite: 3amp_e.
Adding exchange reaction EX_LalaDgluMdapDala_e with default bounds for boundary metabolite: LalaDgluMdapDala_e.
Adding exchange reaction EX_LalaDgluMdap_e with default bounds for boundary metabolite: LalaDgluMdap_e.
Adding exchange reaction EX_acald_e with default bounds for boundary metabolite: acald_e.
Adding exchange reaction EX_acgam1p_e with default bounds for boundary metabolite: acgam1p_e.
Adding exchange reaction EX_acmana_e with default bounds for boundary metabolite: acmana_e.
Adding exchange reaction EX_alaala_e with default bounds for boundary metabolite: alaala_e.
Adding exchange reaction EX_argp_e with default bounds for boundary metabolite: argp_e.
Adding exchange reaction EX_asn__L_e with default bounds for boundary metabolite: asn__L_e.
Adding exchange reaction EX_bz_e with default bounds for b

Found 0 biomass components that cannot be synthesized for model m_638.


Adding exchange reaction EX_2obut_e with default bounds for boundary metabolite: 2obut_e.
Adding exchange reaction EX_6pgc_e with default bounds for boundary metabolite: 6pgc_e.
Adding exchange reaction EX_LalaDgluMdap_e with default bounds for boundary metabolite: LalaDgluMdap_e.
Adding exchange reaction EX_R_3httdca_e with default bounds for boundary metabolite: R_3httdca_e.
Adding exchange reaction EX_ac_e with default bounds for boundary metabolite: ac_e.
Adding exchange reaction EX_acald_e with default bounds for boundary metabolite: acald_e.
Adding exchange reaction EX_acgam1p_e with default bounds for boundary metabolite: acgam1p_e.
Adding exchange reaction EX_acmana_e with default bounds for boundary metabolite: acmana_e.
Adding exchange reaction EX_ala__L_e with default bounds for boundary metabolite: ala__L_e.
Adding exchange reaction EX_alaala_e with default bounds for boundary metabolite: alaala_e.
Adding exchange reaction EX_amp_e with default bounds for boundary metabolit

Found 0 biomass components that cannot be synthesized for model m_939.


### check sucrose stuff

In [42]:
m2774 = read_sbml_model("/home/emma/Dokumente/thesis/Model_generation_curation/Curated_models/mb1_mdr_rdr_dp_lib/1350_or_mb1_mdr_rdr_dp_lib.xml")

In [43]:
s = m2774.slim_optimize()
print(s)

25.865323581532675


In [44]:
m2774.reactions.SUCpts

AttributeError: DictList has no attribute or entry SUCpts